# Kubernetes for Data Engineering — Pods, PVCs, Secrets, Spark on K8s

## Mental Model

In a real data platform, **Kubernetes is the scheduler for data compute**:

- **Pods** run one unit of work or one service replica.
- **ConfigMaps** hold non-secret runtime configuration.
- **Secrets** hold credentials.
- **PersistentVolumeClaims (PVCs)** provide storage that survives pod restarts.
- **Deployments** keep services alive with a target replica count.
- **Services** provide a stable network identity for pods.
- **Spark on Kubernetes** launches a **driver pod** and then dynamic **executor pods**.
- **Airflow with KubernetesExecutor** launches a fresh pod per task, then removes it when the task finishes.

This notebook is grounded in the Citi-style telemetry story from the uploaded spec: PostgreSQL hosts telemetry tables for endpoints, metrics, and alerts; Kubernetes is used as the orchestration substrate for data workloads and checks.


In [ ]:

import base64
import json
import os
import shutil
import subprocess
import sys
import tempfile
import textwrap
import time
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", os.getcwd())

def run_cmd(cmd, input_text=None, check=False, timeout=120):
    """
    UTF-8 safe subprocess wrapper.
    Returns a dict with returncode/stdout/stderr/succeeded.
    """
    if isinstance(cmd, str):
        shell = True
    else:
        shell = False

    try:
        completed = subprocess.run(
            cmd,
            input=input_text,
            text=True,
            capture_output=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
            shell=shell,
            check=False,
        )
        result = {
            "cmd": cmd,
            "returncode": completed.returncode,
            "stdout": completed.stdout,
            "stderr": completed.stderr,
            "succeeded": completed.returncode == 0,
        }
        if check and completed.returncode != 0:
            raise RuntimeError(
                f"Command failed: {cmd}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}"
            )
        return result
    except Exception as exc:
        return {
            "cmd": cmd,
            "returncode": -1,
            "stdout": "",
            "stderr": str(exc),
            "succeeded": False,
        }

kubectl_path = shutil.which("kubectl")
print("kubectl path:", kubectl_path or "NOT FOUND")

cluster_probe = run_cmd([kubectl_path or "kubectl", "cluster-info"], timeout=30)
k8s_available = bool(kubectl_path) and cluster_probe["succeeded"]

if not k8s_available:
    print("Enable Kubernetes in Docker Desktop → Settings → Kubernetes → Enable Kubernetes → Apply")
    if cluster_probe["stderr"]:
        print(cluster_probe["stderr"][:1000])
else:
    print(cluster_probe["stdout"][:2000])

print("k8s_available =", k8s_available)

PG_HOST = "localhost"
PG_PORT = 5432
PG_DB = "de_telemetry"
PG_USER = "de_admin"
PG_PASSWORD = "DeAdmin2026!"

DATASET_CONTEXT = {
    "endpoints_rows": 10000,
    "metrics_rows": 500000,
    "alerts_rows": 25000,
    "narrative": "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers",
}
print(json.dumps(DATASET_CONTEXT, indent=2))


## Optional local PostgreSQL sanity check

This is the same logical health check our Kubernetes demo pods will use: can we reach the local telemetry database, and can we read a few row counts safely?


In [ ]:

postgres_check_summary = {"driver": None, "connected": False, "counts": {}, "error": None}

try:
    import psycopg2  # pre-installed per exercise assumptions
    postgres_check_summary["driver"] = "psycopg2"

    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        connect_timeout=5,
    )
    conn.autocommit = True

    with conn.cursor() as cur:
        for table_name in ["endpoints", "metrics", "alerts"]:
            cur.execute(f"select count(*) from {table_name}")
            postgres_check_summary["counts"][table_name] = cur.fetchone()[0]

    conn.close()
    postgres_check_summary["connected"] = True
except Exception as exc:
    postgres_check_summary["error"] = str(exc)

print(json.dumps(postgres_check_summary, indent=2))


## Setup helpers for Kubernetes sections

Everything below is wrapped in `if k8s_available:` so the notebook stays safe on machines where Docker Desktop Kubernetes is not enabled.


In [ ]:

def kube_apply_yaml(yaml_text):
    return run_cmd([kubectl_path, "apply", "-f", "-"], input_text=yaml_text, timeout=120)

def kube_delete_yaml(yaml_text):
    return run_cmd([kubectl_path, "delete", "-f", "-", "--ignore-not-found=true"], input_text=yaml_text, timeout=120)

def kube_get(kind, name=None, extra_args=None, timeout=60):
    cmd = [kubectl_path, "get", kind]
    if name:
        cmd.append(name)
    if extra_args:
        cmd.extend(extra_args)
    return run_cmd(cmd, timeout=timeout)

def kube_describe(kind, name, timeout=60):
    return run_cmd([kubectl_path, "describe", kind, name], timeout=timeout)

def kube_logs(name, timeout=60):
    return run_cmd([kubectl_path, "logs", name], timeout=timeout)

def wait_for_pod_phase(pod_name, target_phase="Succeeded", timeout_seconds=180):
    start = time.time()
    while time.time() - start < timeout_seconds:
        result = kube_get("pod", pod_name, ["-o", "json"], timeout=30)
        if result["succeeded"]:
            try:
                pod_json = json.loads(result["stdout"])
                phase = pod_json["status"].get("phase")
                if phase == target_phase:
                    return True, phase, result
                if phase in {"Failed", "Unknown"}:
                    return False, phase, result
            except Exception:
                pass
        time.sleep(3)
    return False, "Timeout", None

def wait_for_deployment_ready(name, timeout_seconds=180):
    start = time.time()
    while time.time() - start < timeout_seconds:
        result = kube_get("deployment", name, ["-o", "json"], timeout=30)
        if result["succeeded"]:
            try:
                dep = json.loads(result["stdout"])
                desired = dep["spec"]["replicas"]
                ready = dep.get("status", {}).get("readyReplicas", 0)
                available = dep.get("status", {}).get("availableReplicas", 0)
                if ready >= desired and available >= desired:
                    return True, {"desired": desired, "ready": ready, "available": available}
            except Exception:
                pass
        time.sleep(3)
    return False, {"desired": None, "ready": None, "available": None}


## 1) Core Objects Demo — ConfigMap + Secret


In [ ]:

if k8s_available:
    configmap_yaml = textwrap.dedent(f"""
    apiVersion: v1
    kind: ConfigMap
    metadata:
      name: citi-postgres-config
    data:
      PGHOST: "{PG_HOST}"
      PGPORT: "{PG_PORT}"
      PGDATABASE: "{PG_DB}"
      DATASET_NARRATIVE: "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers"
    """)

    secret_yaml = textwrap.dedent(f"""
    apiVersion: v1
    kind: Secret
    metadata:
      name: citi-postgres-secret
    type: Opaque
    data:
      PGUSER: "{base64.b64encode(PG_USER.encode('utf-8')).decode('utf-8')}"
      PGPASSWORD: "{base64.b64encode(PG_PASSWORD.encode('utf-8')).decode('utf-8')}"
    """)

    print(kube_apply_yaml(configmap_yaml)["stdout"])
    print(kube_apply_yaml(secret_yaml)["stdout"])

    cm_desc = kube_describe("configmap", "citi-postgres-config")
    secret_desc = kube_describe("secret", "citi-postgres-secret")

    print("---- ConfigMap describe ----")
    print(cm_desc["stdout"][:4000] if cm_desc["succeeded"] else cm_desc["stderr"])
    print("---- Secret describe ----")
    print(secret_desc["stdout"][:4000] if secret_desc["succeeded"] else secret_desc["stderr"])
    print("ConfigMap and Secret created")
else:
    print("Skipping Core Objects Demo because k8s_available=False")


## 2) PersistentVolumeClaim Demo

We create a `1Gi` PVC, mount it into a writer pod, write a test file, then mount the same PVC into a reader pod and read the file back. Finally, we clean up.


In [ ]:

if k8s_available:
    pvc_and_writer_yaml = textwrap.dedent("""
    apiVersion: v1
    kind: PersistentVolumeClaim
    metadata:
      name: citi-demo-pvc
    spec:
      accessModes:
        - ReadWriteOnce
      resources:
        requests:
          storage: 1Gi
    ---
    apiVersion: v1
    kind: Pod
    metadata:
      name: citi-pvc-writer
    spec:
      restartPolicy: Never
      containers:
        - name: writer
          image: busybox:1.36
          command:
            - sh
            - -c
            - |
              mkdir -p /data/demo
              echo "PVC write test from Citi DE notebook at $(date -u +%Y-%m-%dT%H:%M:%SZ)" > /data/demo/check.txt
              cat /data/demo/check.txt
          volumeMounts:
            - name: demo-vol
              mountPath: /data
      volumes:
        - name: demo-vol
          persistentVolumeClaim:
            claimName: citi-demo-pvc
    """)

    print(kube_apply_yaml(pvc_and_writer_yaml)["stdout"])

    ok, phase, _ = wait_for_pod_phase("citi-pvc-writer", "Succeeded", timeout_seconds=180)
    print("Writer pod phase:", phase)

    writer_logs = kube_logs("citi-pvc-writer")
    print("---- writer pod logs ----")
    print(writer_logs["stdout"] if writer_logs["succeeded"] else writer_logs["stderr"])

    reader_yaml = textwrap.dedent("""
    apiVersion: v1
    kind: Pod
    metadata:
      name: citi-pvc-reader
    spec:
      restartPolicy: Never
      containers:
        - name: reader
          image: busybox:1.36
          command:
            - sh
            - -c
            - |
              echo "Reading back file from PVC:"
              cat /data/demo/check.txt
          volumeMounts:
            - name: demo-vol
              mountPath: /data
      volumes:
        - name: demo-vol
          persistentVolumeClaim:
            claimName: citi-demo-pvc
    """)

    print(kube_apply_yaml(reader_yaml)["stdout"])

    ok2, phase2, _ = wait_for_pod_phase("citi-pvc-reader", "Succeeded", timeout_seconds=180)
    print("Reader pod phase:", phase2)

    reader_logs = kube_logs("citi-pvc-reader")
    print("---- reader pod logs ----")
    print(reader_logs["stdout"] if reader_logs["succeeded"] else reader_logs["stderr"])

    cleanup_yaml = textwrap.dedent("""
    apiVersion: v1
    kind: Pod
    metadata:
      name: citi-pvc-writer
    ---
    apiVersion: v1
    kind: Pod
    metadata:
      name: citi-pvc-reader
    ---
    apiVersion: v1
    kind: PersistentVolumeClaim
    metadata:
      name: citi-demo-pvc
    """)
    print(kube_delete_yaml(cleanup_yaml)["stdout"])
else:
    print("Skipping PVC Demo because k8s_available=False")


## 3) DE Deployment Demo — `citi-checker`

This creates a 2-replica Deployment that uses the local PostgreSQL endpoint check pattern. The container waits until the database is reachable via `pg_isready`, then stays alive. A Service is created so we can inspect Kubernetes endpoints for the deployment.


In [ ]:

if k8s_available:
    deployment_yaml = textwrap.dedent(f"""
    apiVersion: apps/v1
    kind: Deployment
    metadata:
      name: citi-checker
    spec:
      replicas: 2
      selector:
        matchLabels:
          app: citi-checker
      template:
        metadata:
          labels:
            app: citi-checker
        spec:
          containers:
            - name: checker
              image: postgres:16-alpine
              envFrom:
                - configMapRef:
                    name: citi-postgres-config
                - secretRef:
                    name: citi-postgres-secret
              command:
                - sh
                - -c
                - |
                  echo "Starting postgres reachability checks..."
                  for i in $(seq 1 30); do
                    pg_isready -h "$PGHOST" -p "$PGPORT" -U "$PGUSER" -d "$PGDATABASE" && break
                    echo "Waiting for postgres..."
                    sleep 2
                  done
                  pg_isready -h "$PGHOST" -p "$PGPORT" -U "$PGUSER" -d "$PGDATABASE"
                  echo "healthy"
                  while true; do sleep 3600; done
              ports:
                - containerPort: 5432
    ---
    apiVersion: v1
    kind: Service
    metadata:
      name: citi-checker-svc
    spec:
      selector:
        app: citi-checker
      ports:
        - protocol: TCP
          port: 5432
          targetPort: 5432
    """)

    print(kube_apply_yaml(deployment_yaml)["stdout"])

    ready, deploy_state = wait_for_deployment_ready("citi-checker", timeout_seconds=240)
    print("Deployment ready:", ready, deploy_state)

    pods_result = kube_get("pods", extra_args=["-l", "app=citi-checker", "-o", "wide"], timeout=60)
    endpoints_result = kube_get("endpoints", "citi-checker-svc", ["-o", "wide"], timeout=60)

    print("---- pods ----")
    print(pods_result["stdout"] if pods_result["succeeded"] else pods_result["stderr"])

    print("---- service endpoints ----")
    print(endpoints_result["stdout"] if endpoints_result["succeeded"] else endpoints_result["stderr"])

    if ready:
        print("Deployment running, 2 replicas healthy")
    else:
        print("Deployment did not fully reach ready state before timeout")
else:
    print("Skipping Deployment Demo because k8s_available=False")


## 4) Spark on K8s Architecture

### What actually happens

When Spark runs on Kubernetes:

1. **spark-submit** talks to the Kubernetes API.
2. Spark creates a **driver pod**.
3. The driver requests **executor pods** dynamically.
4. Executors scale up and down with workload demand.
5. Checkpoints, shuffle spill, or durable intermediate state can be directed to **PVC-backed storage** or object storage, depending on the design.

### Spark submit shape for Kubernetes

```bash
spark-submit \
  --master k8s://https://kubernetes.docker.internal:6443 \
  --deploy-mode cluster \
  --name citi-telemetry-job \
  --class org.apache.spark.examples.SparkPi \
  --conf spark.executor.instances=2 \
  --conf spark.kubernetes.container.image=bitnami/spark:3.5.4 \
  --conf spark.kubernetes.namespace=default \
  --conf spark.kubernetes.authenticate.driver.serviceAccountName=default \
  local:///opt/spark/examples/jars/spark-examples_2.12-3.5.4.jar
```

### Why PVCs matter for DE

For data engineering, PVCs are useful when you need storage that **survives pod termination**:

- local checkpoint directories
- staging files
- intermediate artifacts
- restart-safe scratch space in local demos

In production, many teams still prefer object storage for durability, but PVCs are the cleanest local Docker Desktop demonstration of persistent state in Kubernetes.


## 5) What Just Happened

Every Spark executor in production Citi is a K8s pod. The KubernetesExecutor in Airflow spins a pod per task and deletes it when done. PVCs provide the durable checkpoint storage that survives pod termination.

That is the key platform idea:

- **Kubernetes gives you disposable compute**
- **PVCs give you durable local state**
- **ConfigMaps and Secrets inject runtime configuration safely**
- **Deployments keep long-running services healthy**
